## Master Class 4

### Task 1 — Design a Class

In [1]:
class TrainingRun:
    run_count = 0

    def __init__(self, model_name, learning_rate):
        if not self.validate_learning_rate(learning_rate):
            raise ValueError("Learning rate must be between 0 and 1.")

        self.model_name = model_name
        self.learning_rate = learning_rate
        self.status = "Not Started"

        TrainingRun.run_count += 1

    def start(self):
        self.status = "Running"
        print(f"{self.model_name} training started.")

    def summary(self):
        return (
            f"Model: {self.model_name}, "
            f"Learning Rate: {self.learning_rate}, "
            f"Status: {self.status}"
        )

    @classmethod
    def from_config(cls, config_dict):
        return cls(
            config_dict["model_name"],
            config_dict["learning_rate"]
        )

    @staticmethod
    def validate_learning_rate(learning_rate):
        return 0 < learning_rate < 1


run1 = TrainingRun("CNN", 0.01)
run2 = TrainingRun("ResNet", 0.001)
run3 = TrainingRun("LSTM", 0.05)
print("Total runs:", TrainingRun.run_count)

run1.start()

print(run1.summary())

config = {
    "model_name": "Transformer",
    "learning_rate": 0.0001
}
run4 = TrainingRun.from_config(config)

print(run4.summary())
print("Total runs:", TrainingRun.run_count)

Total runs: 3
CNN training started.
Model: CNN, Learning Rate: 0.01, Status: Running
Model: Transformer, Learning Rate: 0.0001, Status: Not Started
Total runs: 4


### Task 2 — Encapsulation

In [2]:
class TrainingRun:
    run_count = 0

    def __init__(self, model_name, learning_rate):
        if not self.validate_learning_rate(learning_rate):
            raise ValueError("Learning rate must be between 0 and 1.")

        self.model_name = model_name
        self.learning_rate = learning_rate
        self._status = "pending"
        self.__api_key = "fake-training-token-123"

        TrainingRun.run_count += 1

    def start(self):
        self.status = "running"
        print(f"{self.model_name} training started.")

    def summary(self):
        return (
            f"Model: {self.model_name}, "
            f"Learning Rate: {self.learning_rate}, "
            f"Status: {self.status}"
        )

    @classmethod
    def from_config(cls, config_dict):
        return cls(
            config_dict["model_name"],
            config_dict["learning_rate"]
        )

    @staticmethod
    def validate_learning_rate(learning_rate):
        return 0 < learning_rate < 1

    @property
    def status(self):
        return self._status

    @status.setter
    def status(self, value):
        if value not in ["pending", "running", "done"]:
            raise ValueError(
                "Status must be 'pending', 'running', or 'done'."
            )

        self._status = value


run = TrainingRun("CNN", 0.01)
print(run.status)

run.status = "running"
print(run.status)

run.status = "done"
print(run.status)


# Invalid status
# This will raise ValueError
# run.status = "paused"


# Private attribute cannot be accessed directly
# This will raise AttributeError
# print(run.__api_key)


# But it can still be accessed using Python's name-mangling
print(run._TrainingRun__api_key)

pending
running
done
fake-training-token-123


### Task 3 — Inheritance

In [3]:
class LRSchedulerRun(TrainingRun):

    def __init__(self, model_name, learning_rate, schedule):
        super().__init__(model_name, learning_rate)
        self.schedule = schedule

    def summary(self):
        parent_summary = super().summary()
        current_learning_rate = self.schedule[0]

        return (
            f"{parent_summary}, "
            f"Current Scheduled Learning Rate: {current_learning_rate}"
        )


run = LRSchedulerRun(
    "CNN",
    0.01,
    [0.01, 0.005, 0.001]
)

print(run.summary())
print("Total runs:", TrainingRun.run_count)

Model: CNN, Learning Rate: 0.01, Status: pending, Current Scheduled Learning Rate: 0.01
Total runs: 2


### Task 4 — Polymorphism

In [4]:
class EarlyStoppingRun(TrainingRun):

    def __init__(self, model_name, learning_rate, patience):
        super().__init__(model_name, learning_rate)
        self.patience = patience

    def summary(self):
        parent_summary = super().summary()
        return f"{parent_summary}, Patience: {self.patience} epochs"


def print_all_summaries(runs):
    for run in runs:
        print(run.summary())


run1 = TrainingRun("CNN", 0.01)

run2 = LRSchedulerRun(
    "ResNet",
    0.001,
    [0.001, 0.0005, 0.0001]
)

run3 = EarlyStoppingRun(
    "LSTM",
    0.005,
    5
)

runs = [run1, run2, run3]

print_all_summaries(runs)

Model: CNN, Learning Rate: 0.01, Status: pending
Model: ResNet, Learning Rate: 0.001, Status: pending, Current Scheduled Learning Rate: 0.001
Model: LSTM, Learning Rate: 0.005, Status: pending, Patience: 5 epochs


### Task 5 — Duck Typing vs. Interfaces

In [5]:
from abc import ABC, abstractmethod


# Duck typing

class Cleaner:
    def process(self, data):
        return data.strip()


class Tokenizer:
    def process(self, data):
        return data.split()


class Normalizer:
    def process(self, data):
        return data.lower()


def run_pipeline(steps, data):
    for step in steps:
        data = step.process(data)
    return data


steps = [Cleaner(), Normalizer(), Tokenizer()]

result = run_pipeline(steps, "  Hello Python World  ")
print(result)


# ABC version

class Step(ABC):

    @abstractmethod
    def process(self, data):
        pass


class CleanerStep(Step):
    def process(self, data):
        return data.strip()


class NormalizerStep(Step):
    def process(self, data):
        return data.lower()


class TokenizerStep(Step):
    def process(self, data):
        return data.split()


def run_pipeline_abc(steps, data):
    for step in steps:
        data = step.process(data)
    return data


steps_abc = [
    CleanerStep(),
    NormalizerStep(),
    TokenizerStep()
]

result = run_pipeline_abc(steps_abc, "  Hello Python World  ")
print(result)


# Step() cannot be instantiated
# This raises TypeError:
# step = Step()

['hello', 'python', 'world']
['hello', 'python', 'world']


### Task 6 — Dunder Methods

In [6]:
class TrainingRun:
    run_count = 0

    def __init__(self, model_name, learning_rate):
        if not self.validate_learning_rate(learning_rate):
            raise ValueError("Learning rate must be between 0 and 1.")

        self.model_name = model_name
        self.learning_rate = learning_rate
        self.status = "pending"
        self.__api_key = "fake-training-token-123"

        TrainingRun.run_count += 1

    def start(self):
        self.status = "running"
        print(f"{self.model_name} training started.")

    def summary(self):
        return (
            f"Model: {self.model_name}, "
            f"Learning Rate: {self.learning_rate}, "
            f"Status: {self.status}"
        )

    @classmethod
    def from_config(cls, config_dict):
        return cls(
            config_dict["model_name"],
            config_dict["learning_rate"]
        )

    @staticmethod
    def validate_learning_rate(learning_rate):
        return 0 < learning_rate < 1

    @property
    def status(self):
        return self._status

    @status.setter
    def status(self, value):
        if value not in ["pending", "running", "done"]:
            raise ValueError(
                "Status must be 'pending', 'running', or 'done'."
            )
        self._status = value

    def __str__(self):
        return f"TrainingRun(model='{self.model_name}', lr={self.learning_rate})"

    def __eq__(self, other):
        if not isinstance(other, TrainingRun):
            return False

        return (
            self.model_name == other.model_name
            and self.learning_rate == other.learning_rate
        )

    def __repr__(self):
        return (
            f"TrainingRun("
            f"model_name='{self.model_name}', "
            f"learning_rate={self.learning_rate})"
        )


run1 = TrainingRun("gpt-mini", 0.01)
run2 = TrainingRun("gpt-mini", 0.01)
run3 = TrainingRun("CNN", 0.02)

print(run1)

print(run1 == run2)
print(run1 == run3)

print(repr(run1))

TrainingRun(model='gpt-mini', lr=0.01)
True
False
TrainingRun(model_name='gpt-mini', learning_rate=0.01)
